<a href="https://colab.research.google.com/github/rickyajs/Data-Science-2026/blob/main/Pertemuan_10_RickyArmandaJayaSirait_240401020219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nama  : Ricky Armanda Jaya Sirait
# Nim   : 240401020219
#Kelas  : IF401
#Prodi  : PJJ S1 Informatika

In [2]:
import pandas as pd

# Mengunduh dataset Telco Customer
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Menampilkan dimensi/ukuran dataset
print("=== DIMENSI DATA ===")
print(f"Dimensi dataset: {df.shape}\n")

# Menghitung proporsi kelas churn untuk memastikan imbalanced dataset
print("=== PROPORSI KELAS TARGET (CHURN) ===")
print(df["Churn"].value_counts(normalize=True))
print("-" * 50)


from sklearn.model_selection import train_test_split

# Pembersihan Data: Mengubah 'TotalCharges' yang kosong (berisi spasi) menjadi NaN dan menghapusnya
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

# Menghapus kolom 'customerID' karena tidak memiliki nilai prediktif
X = df.drop(columns=['customerID', 'Churn'])

# Mengubah target 'Churn' menjadi numerik (Yes = 1, No = 0)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Encoding fitur kategorikal menggunakan One-Hot Encoding (pd.get_dummies)
X = pd.get_dummies(X, drop_first=True)

# Membagi data menjadi data latih (80%) dan data uji (20%) secara stratified
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("=== PREPROCESSING SELESAI ===")
print(f"Jumlah data latihan (X_tr): {X_tr.shape[0]} baris")
print(f"Jumlah data pengujian (X_te): {X_te.shape[0]} baris\n")
print("-" * 50)


from sklearn.ensemble import RandomForestClassifier

# Membangun model RandomForestClassifier dengan penanganan class_weight="balanced"
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)

# Melatih model menggunakan data latihan
rf.fit(X_tr, y_tr)

print("=== MODEL BERHASIL DILATIH ===\n")
print("-" * 50)


from sklearn.metrics import classification_report, roc_auc_score

# Menghitung prediksi label (0 atau 1)
y_pred = rf.predict(X_te)

# Menghitung probabilitas untuk kelas positif (Churn = 1)
proba = rf.predict_proba(X_te)[:, 1]

print("=== EVALUASI MODEL ===")
print("Classification Report:")
print(classification_report(y_te, y_pred))

# Menghitung nilai ROC-AUC
roc_auc = roc_auc_score(y_te, proba)
print(f"ROC-AUC Score: {roc_auc:.4f}\n")
print("-" * 50)


print("=== LANGKAH 5: KESIMPULAN UTAMA MODEL ===")

# Menampilkan 5 sampel pertama hasil prediksi probabilitas churn pelanggan
sample_proba = pd.DataFrame({
    'Aktual Churn': y_te.head().values,
    'Prediksi Label': y_pred[:5],
    'Probabilitas Churn (%)': (proba[:5] * 100).round(2)
})
print("Contoh Hasil Prediksi Probabilitas Churn:")
print(sample_proba.to_string(index=False))
print("\n[Kesimpulan Analisis]:")
print("1. Dataset ini terbukti mengalami ketidakseimbangan kelas (imbalanced) dengan proporsi kelas minoritas (Churn) sebesar ~26,5%.")
print("2. Penggunaan parameter 'class_weight=\"balanced\"' pada algoritma Random Forest berhasil membantu model memberikan perhatian ekstra pada kelas minoritas tanpa harus memanipulasi distribusi data.")
print("3. Berdasarkan hasil evaluasi, model menghasilkan nilai ROC-AUC yang solid, menandakan kemampuan pemeringkatan probabilitas risiko churn pelanggan yang cukup akurat.")
print("4. Nilai recall pada kelas minoritas (1) perlu dipantau secara berkala untuk memastikan tim retensi perusahaan dapat memprioritaskan tindakan pencegahan secara maksimal terhadap pelanggan yang berisiko tinggi.")

=== DIMENSI DATA ===
Dimensi dataset: (7043, 21)

=== PROPORSI KELAS TARGET (CHURN) ===
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
--------------------------------------------------
=== PREPROCESSING SELESAI ===
Jumlah data latihan (X_tr): 5625 baris
Jumlah data pengujian (X_te): 1407 baris

--------------------------------------------------
=== MODEL BERHASIL DILATIH ===

--------------------------------------------------
=== EVALUASI MODEL ===
Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.63      0.49      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC Score: 0.8199

--------------------------------------------------
=== LANGKAH 5: KESIMPULAN UTAMA MODEL ===
Contoh Hasil Prediksi Probabilitas Churn:
 Aktual Churn  P